In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

from sklearn.neural_network import MLPClassifier
from sklearn.preproces.sing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

train = pd.read_csv(
    "/content/drive/MyDrive/Fraud_Project/E1_train.csv"
)

validation = pd.read_csv(
    "/content/drive/MyDrive/Fraud_Project/E1_validation.csv"
)

test = pd.read_csv(
    "/content/drive/MyDrive/Fraud_Project/E1_test.csv"
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (18734, 23)
Validation: (6322, 23)
Test: (6185, 23)


In [ ]:
def create_financial_features(df):
    df = df.copy()

    df["current_ratio"] = (
        df["AssetsCurrent"] /
        df["LiabilitiesCurrent"].replace(0, np.nan)
    )

    df["cash_ratio"] = (
        df["CashAndCashEquivalentsAtCarryingValue"] /
        df["LiabilitiesCurrent"].replace(0, np.nan)
    )

    df["current_assets_ratio"] = (
        df["AssetsCurrent"] /
        df["Assets"].replace(0, np.nan)
    )

    df["liabilities_to_assets"] = (
        df["Liabilities"] /
        df["Assets"].replace(0, np.nan)
    )

    df["debt_to_equity"] = (
        df["Liabilities"] /
        df["StockholdersEquity"].replace(0, np.nan)
    )

    df["roa"] = (
        df["NetIncomeLoss"] /
        df["Assets"].replace(0, np.nan)
    )

    df["roe"] = (
        df["NetIncomeLoss"] /
        df["StockholdersEquity"].replace(0, np.nan)
    )

    df["operating_return_on_assets"] = (
        df["OperatingIncomeLoss"] /
        df["Assets"].replace(0, np.nan)
    )

    return df


train = create_financial_features(train)
validation = create_financial_features(validation)
test = create_financial_features(test)

In [ ]:
financial_features = [
    "price",
    "Assets",
    "AssetsCurrent",
    "CashAndCashEquivalentsAtCarryingValue",
    "CommonStockSharesIssued",
    "CommonStockValue",
    "IncomeTaxExpenseBenefit",
    "LiabilitiesCurrent",
    "OperatingIncomeLoss",
    "StockholdersEquity",
    "NetIncomeLoss",
    "Liabilities",
    "current_ratio",
    "cash_ratio",
    "current_assets_ratio",
    "liabilities_to_assets",
    "debt_to_equity",
    "roa",
    "roe",
    "operating_return_on_assets"
]

X_train = train[financial_features].copy()
X_val = validation[financial_features].copy()
X_test = test[financial_features].copy()

y_train = train["is_fraud"].copy()
y_val = validation["is_fraud"].copy()
y_test = test["is_fraud"].copy()

print("Features:", len(financial_features))
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Features: 20
X_train: (18734, 20)
X_val: (6322, 20)
X_test: (6185, 20)


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------
# 1. Replace infinite values with NaN
# --------------------------------------------------

X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)


# --------------------------------------------------
# 2. Median imputation
#    FIT ONLY on training data
# --------------------------------------------------

mlp_imputer = SimpleImputer(strategy="median")

X_train_imp = pd.DataFrame(
    mlp_imputer.fit_transform(X_train),
    columns=financial_features,
    index=X_train.index
)

X_val_imp = pd.DataFrame(
    mlp_imputer.transform(X_val),
    columns=financial_features,
    index=X_val.index
)

X_test_imp = pd.DataFrame(
    mlp_imputer.transform(X_test),
    columns=financial_features,
    index=X_test.index
)


# --------------------------------------------------
# 3. Standardization
#    FIT ONLY on training data
# --------------------------------------------------

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    columns=financial_features,
    index=X_train_imp.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val_imp),
    columns=financial_features,
    index=X_val_imp.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imp),
    columns=financial_features,
    index=X_test_imp.index
)


# --------------------------------------------------
# 4. Check preprocessing
# --------------------------------------------------

print("Missing values after imputation:")
print("Train:", X_train_scaled.isna().sum().sum())
print("Validation:", X_val_scaled.isna().sum().sum())
print("Test:", X_test_scaled.isna().sum().sum())

print("\nTraining feature means:")
print(X_train_scaled.mean().round(4))

print("\nTraining feature standard deviations:")
print(X_train_scaled.std().round(4))

Missing values after imputation:
Train: 0
Validation: 0
Test: 0

Training feature means:
price                                   -0.0
Assets                                  -0.0
AssetsCurrent                           -0.0
CashAndCashEquivalentsAtCarryingValue    0.0
CommonStockSharesIssued                  0.0
CommonStockValue                         0.0
IncomeTaxExpenseBenefit                  0.0
LiabilitiesCurrent                      -0.0
OperatingIncomeLoss                      0.0
StockholdersEquity                      -0.0
NetIncomeLoss                           -0.0
Liabilities                              0.0
current_ratio                           -0.0
cash_ratio                               0.0
current_assets_ratio                     0.0
liabilities_to_assets                   -0.0
debt_to_equity                          -0.0
roa                                     -0.0
roe                                     -0.0
operating_return_on_assets              -0.0
dtype: floa

In [ ]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42
)

mlp_model.fit(
    X_train_scaled,
    y_train
)

print("MLP training completed.")
print("Iterations:", mlp_model.n_iter_)

MLP training completed.
Iterations: 50


In [ ]:
# Validation probabilities
y_val_prob_mlp = mlp_model.predict_proba(X_val_scaled)[:, 1]

# Default threshold
y_val_pred_mlp = (y_val_prob_mlp >= 0.5).astype(int)

print("MLP Validation Results")
print("-" * 40)

print("Precision :", precision_score(
    y_val, y_val_pred_mlp, zero_division=0
))

print("Recall    :", recall_score(
    y_val, y_val_pred_mlp, zero_division=0
))

print("F1-score  :", f1_score(
    y_val, y_val_pred_mlp, zero_division=0
))

print("ROC-AUC   :", roc_auc_score(
    y_val, y_val_prob_mlp
))

print("PR-AUC    :", average_precision_score(
    y_val, y_val_prob_mlp
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_mlp))

MLP Validation Results
----------------------------------------
Precision : 0.0
Recall    : 0.0
F1-score  : 0.0
ROC-AUC   : 0.6382016493748337
PR-AUC    : 0.020413516831085703

Confusion Matrix:
[[6265    0]
 [  57    0]]


In [ ]:
thresholds = [0.50, 0.30, 0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.001]

print("MLP Threshold Analysis")
print("-" * 75)
print(
    f"{'Threshold':<12}"
    f"{'Predicted Fraud':<18}"
    f"{'Precision':<15}"
    f"{'Recall':<15}"
    f"{'F1':<10}"
)

for threshold in thresholds:

    y_pred = (y_val_prob_mlp >= threshold).astype(int)

    precision = precision_score(
        y_val, y_pred, zero_division=0
    )

    recall = recall_score(
        y_val, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_val, y_pred, zero_division=0
    )

    print(
        f"{threshold:<12}"
        f"{y_pred.sum():<18}"
        f"{precision:<15.4f}"
        f"{recall:<15.4f}"
        f"{f1:<10.4f}"
    )

MLP Threshold Analysis
---------------------------------------------------------------------------
Threshold   Predicted Fraud   Precision      Recall         F1        
0.5         0                 0.0000         0.0000         0.0000    
0.3         0                 0.0000         0.0000         0.0000    
0.2         2                 0.0000         0.0000         0.0000    
0.1         8                 0.0000         0.0000         0.0000    
0.05        70                0.0143         0.0175         0.0157    
0.02        521               0.0365         0.3333         0.0657    
0.01        1206              0.0191         0.4035         0.0364    
0.005       3966              0.0111         0.7719         0.0219    
0.001       6163              0.0088         0.9474         0.0174    


In [ ]:
print("MLP Validation Probability Statistics")
print("-" * 50)

print("Minimum :", y_val_prob_mlp.min())
print("Maximum :", y_val_prob_mlp.max())
print("Mean    :", y_val_prob_mlp.mean())
print("Median  :", np.median(y_val_prob_mlp))

print("\nFraud cases:")
print(
    pd.Series(y_val_prob_mlp[y_val.values == 1]).describe()
)

print("\nNon-fraud cases:")
print(
    pd.Series(y_val_prob_mlp[y_val.values == 0]).describe()
)

MLP Validation Probability Statistics
--------------------------------------------------
Minimum : 3.9723674075418816e-140
Maximum : 0.24737465865858507
Mean    : 0.008362369587640719
Median  : 0.005585852026681928

Fraud cases:
count    57.000000
mean      0.014158
std       0.012977
min       0.000704
25%       0.005063
50%       0.007868
75%       0.022500
max       0.050430
dtype: float64

Non-fraud cases:
count     6.265000e+03
mean      8.309636e-03
std       1.028089e-02
min      3.972367e-140
25%       4.229187e-03
50%       5.582289e-03
75%       8.050297e-03
max       2.473747e-01
dtype: float64


In [ ]:
mlp_configs = {
    "MLP_A": {
        "hidden_layer_sizes": (32, 16),
        "alpha": 0.0001,
        "learning_rate_init": 0.001
    },

    "MLP_B": {
        "hidden_layer_sizes": (64, 32),
        "alpha": 0.0001,
        "learning_rate_init": 0.001
    },

    "MLP_C": {
        "hidden_layer_sizes": (128, 64),
        "alpha": 0.0001,
        "learning_rate_init": 0.001
    },

    "MLP_D": {
        "hidden_layer_sizes": (64, 32),
        "alpha": 0.001,
        "learning_rate_init": 0.001
    }
}

mlp_results = []

for name, params in mlp_configs.items():

    model = MLPClassifier(
        hidden_layer_sizes=params["hidden_layer_sizes"],
        activation="relu",
        solver="adam",
        alpha=params["alpha"],
        learning_rate_init=params["learning_rate_init"],
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=42
    )

    model.fit(
        X_train_scaled,
        y_train
    )

    val_prob = model.predict_proba(X_val_scaled)[:, 1]

    mlp_results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_val, val_prob),
        "PR-AUC": average_precision_score(y_val, val_prob),
        "Iterations": model.n_iter_
    })

mlp_results_df = pd.DataFrame(mlp_results).sort_values(
    "PR-AUC",
    ascending=False
)

print(mlp_results_df.to_string(index=False))

Model  ROC-AUC   PR-AUC  Iterations
MLP_D 0.650470 0.024581          46
MLP_B 0.638202 0.020414          50
MLP_A 0.379275 0.011061          22
MLP_C 0.487336 0.009818          22


In [ ]:
final_mlp_e1 = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42
)

final_mlp_e1.fit(
    X_train_scaled,
    y_train
)

print("Selected MLP_D trained.")
print("Iterations:", final_mlp_e1.n_iter_)

Selected MLP_D trained.
Iterations: 46


In [ ]:
import joblib

joblib.dump(
    final_mlp_e1,
    "/content/drive/MyDrive/Fraud_Project/E1_MLP_model.pkl"
)

joblib.dump(
    mlp_imputer,
    "/content/drive/MyDrive/Fraud_Project/E1_MLP_imputer.pkl"
)

joblib.dump(
    scaler,
    "/content/drive/MyDrive/Fraud_Project/E1_MLP_scaler.pkl"
)

print("MLP model, imputer, and scaler saved successfully.")

MLP model, imputer, and scaler saved successfully.


In [ ]:
# Validation-selected threshold for MLP
mlp_threshold = 0.02

print("Selected MLP threshold:", mlp_threshold)

Selected MLP threshold: 0.02


In [ ]:
# MLP test probabilities
y_test_prob_mlp = final_mlp_e1.predict_proba(X_test_scaled)[:, 1]

# Apply validation-selected threshold
y_test_pred_mlp = (
    y_test_prob_mlp >= mlp_threshold
).astype(int)


In [ ]:
print("E1 MLP — FINAL TEST RESULTS")
print("-" * 45)

print("Precision :",
      precision_score(y_test, y_test_pred_mlp, zero_division=0))

print("Recall    :",
      recall_score(y_test, y_test_pred_mlp, zero_division=0))

print("F1-score  :",
      f1_score(y_test, y_test_pred_mlp, zero_division=0))

print("ROC-AUC   :",
      roc_auc_score(y_test, y_test_prob_mlp))

print("PR-AUC    :",
      average_precision_score(y_test, y_test_prob_mlp))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_mlp))

E1 MLP — FINAL TEST RESULTS
---------------------------------------------
Precision : 0.02862595419847328
Recall    : 0.234375
F1-score  : 0.05102040816326531
ROC-AUC   : 0.5588241300441105
PR-AUC    : 0.026918925037496898

Confusion Matrix:
[[5612  509]
 [  49   15]]
